# Weight Analysis Notebook

This notebook provides tools to analyze, load, and inspect pretrained model weights from ClimateSAM checkpoints. It includes utilities for examining weight statistics, layer information, and adapter components.

In [11]:
import os
import sys
import torch
import numpy as np
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Project root: C:\Users\perrydebussy\Project\Study\masterarbeit
PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5070


In [12]:
# Setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import model components
try:
    from model.climatesam import ClimateSAM
    print("✓ ClimateSAM imported successfully")
except ImportError as e:
    print(f"✗ Error importing ClimateSAM: {e}")

Using device: cuda:0
✓ ClimateSAM imported successfully


In [13]:
# Initialize model
model_type = 'vit_b'  # Change this to 'vit_l' or 'vit_h' if needed
mlp_ratio = 1.0

try:
    model = ClimateSAM(
        model_type=model_type,
        mlp_ratio=mlp_ratio,
        enable_wandb_logging=False
    ).to(device=device)
    print(f"✓ Model initialized with type: {model_type}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
except Exception as e:
    print(f"✗ Error initializing model: {e}")

✓ Model initialized with type: vit_b
Model parameters: 101,262,723


In [17]:
# Setup checkpoint paths
exp_dir = Path('exp')
# checkpoint_name = 'phase_1_weights_official_vit_b_infused_token_vit_b_mlp1.pth'
checkpoint_name = 'exp_LORA_phase_1_weights_official_vit_b.pth'
image_encoder_path = exp_dir / checkpoint_name

# Verify path exists
if image_encoder_path.exists():
    print(f"✓ Checkpoint found: {image_encoder_path}")
    print(f"  File size: {image_encoder_path.stat().st_size / (1024**2):.2f} MB")
else:
    print(f"✗ Checkpoint not found at: {image_encoder_path}")
    print(f"Available files in {exp_dir}:")
    if exp_dir.exists():
        for f in sorted(exp_dir.glob('*.pth'))[:5]:
            print(f"  - {f.name}")

✓ Checkpoint found: exp\exp_LORA_phase_1_weights_official_vit_b.pth
  File size: 382.22 MB


In [18]:
import os
import torch


# image_encoder_path = "exp\exp_phase_1_weights_official_vit_b_infused_token_vit_b_mlp1.pth"

# Load checkpoint
print("=" * 60)
print("LOADING CHECKPOINT")
print("=" * 60)

try:
    phase_1_checkpoint = torch.load(image_encoder_path, map_location=device)
    print(f"✓ Pretrained weights loaded from {image_encoder_path}")
except Exception as e:
    print(f"✗ Error loading checkpoint: {e}")
    raise

# Print checkpoint structure
print(f"\nCheckpoint keys: {list(phase_1_checkpoint.keys())}")
# print(f"Checkpoint size: {sum(v.numel() for v in phase_1_checkpoint.values()):,} parameters")

# Load image encoder and mask decoder
print("\n" + "=" * 60)
print("LOADING MODEL COMPONENTS")
print("=" * 60)

try:
    model.image_encoder.load_state_dict(phase_1_checkpoint['image_encoder'])
    print("✓ Image encoder weights loaded")
except Exception as e:
    print(f"✗ Error loading image encoder: {e}")

try:
    model.mask_decoder.load_state_dict(phase_1_checkpoint['mask_decoder'])
    print("✓ Mask decoder weights loaded")
except Exception as e:
    print(f"✗ Error loading mask decoder: {e}")

# Load and print input adapter weights if available
print("\n" + "=" * 60)
print("INPUT ADAPTER ANALYSIS")
print("=" * 60)

if 'input_adapter' in phase_1_checkpoint:
    print("\n✓ Input adapter found in checkpoint\n")
    input_adapter_state = phase_1_checkpoint['input_adapter']
    
    total_params = 0
    for param_name, param_value in input_adapter_state.items():
        num_params = param_value.numel()
        total_params += num_params
        print(f"{param_name}:")
        print(f"  Shape: {param_value.shape}")
        print(f"  Dtype: {param_value.dtype}")
        print(f"  Parameters: {num_params:,}")
        print(f"  Mean: {param_value.mean():.6f}")
        print(f"  Std: {param_value.std():.6f}")
        print(f"  Min: {param_value.min():.6f}")
        print(f"  Max: {param_value.max():.6f}\n")
    
    print(f"Total adapter parameters: {total_params:,}")
    
    # Load into model if it has input_adapter attribute
    if hasattr(model, 'input_adapter'):
        try:
            model.input_adapter.load_state_dict(input_adapter_state)
            print("✓ Input adapter weights loaded into model")
        except Exception as e:
            print(f"✗ Error loading input adapter into model: {e}")
    else:
        print("⚠ Model does not have 'input_adapter' attribute")
else:
    print("⚠ Input adapter not found in checkpoint")

LOADING CHECKPOINT
✓ Pretrained weights loaded from exp\exp_LORA_phase_1_weights_official_vit_b.pth

Checkpoint keys: ['image_encoder', 'input_adapter', 'mask_decoder_tc', 'mask_decoder_ar']

LOADING MODEL COMPONENTS
✗ Error loading image encoder: Error(s) in loading state_dict for ClimateSAMImageEncoder:
	Missing key(s) in state_dict: "hq_token_ar", "hq_token_tc", "sam_img_encoder.pos_embed", "sam_img_encoder.patch_embed.proj.weight", "sam_img_encoder.patch_embed.proj.bias", "sam_img_encoder.blocks.0.norm1.weight", "sam_img_encoder.blocks.0.norm1.bias", "sam_img_encoder.blocks.0.attn.rel_pos_h", "sam_img_encoder.blocks.0.attn.rel_pos_w", "sam_img_encoder.blocks.0.attn.qkv.weight", "sam_img_encoder.blocks.0.attn.qkv.bias", "sam_img_encoder.blocks.0.attn.proj.weight", "sam_img_encoder.blocks.0.attn.proj.bias", "sam_img_encoder.blocks.0.norm2.weight", "sam_img_encoder.blocks.0.norm2.bias", "sam_img_encoder.blocks.0.mlp.lin1.weight", "sam_img_encoder.blocks.0.mlp.lin1.bias", "sam_img_enco

In [ ]:
# Print only specific adapter layer weights
if 'input_adapter' in phase_1_checkpoint:
    adapter_weights = phase_1_checkpoint['input_adapter']
    for name, param in adapter_weights.items():
        if 'weight' in name:  # Only print weight parameters, not biases
            print(f"{name}: {param.shape}")
            print(f"  Sample values: {param.flatten()}")  # 
            
            

input_adapt.0.weight: torch.Size([3, 16, 1, 1])
  Sample values: tensor([ 1.0000e+00,  1.0000e+00,  1.0000e+00, -9.3168e-03, -3.3757e-02,
         3.8157e-02,  2.1742e-02, -8.8240e-04,  2.0296e-02, -1.2993e-02,
        -6.0206e-02,  2.0268e-02,  1.6998e-02, -2.4468e-02,  1.5388e-01,
         5.6216e-02,  1.0000e+00,  1.0000e+00,  1.0000e+00,  3.8169e-02,
        -4.6651e-02,  3.5462e-02, -1.3517e-02, -1.0806e-02,  2.9066e-04,
        -2.2958e-02,  8.1592e-02,  4.6016e-03, -2.1108e-02,  8.7190e-02,
        -1.5198e-02, -6.9616e-02,  1.0000e+00,  1.0000e+00,  1.0000e+00,
         1.2920e-02,  7.2600e-02, -2.9548e-02,  9.5846e-03,  1.6739e-02,
         6.4049e-02,  7.0823e-02, -3.9481e-02,  4.0348e-02,  4.6432e-02,
        -8.7386e-03,  8.8214e-03,  7.5384e-03], device='cuda:0')


LORA

In [ ]:
# LoRA Weights Analysis

Analyze LoRA (Low-Rank Adaptation) weights from the phase 1 checkpoint